Taller 5 redes neuronales

In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
%cd /content/drive/MyDrive/Riesgos



In [3]:
!ls


In [4]:
import pandas as pd

df = pd.read_excel('riesgo.xlsx')


In [5]:
df.info()

In [6]:
df.isnull().sum()


In [7]:
df.drop('Type_of_Loan', axis=1, inplace=True)


In [8]:
df.describe().T

In [9]:
df.drop('Name', axis=1, inplace=True)
df.drop('Customer_ID', axis=1, inplace=True)
df.drop('SSN', axis=1, inplace=True)
df.drop('Occupation', axis=1, inplace=True)
df.drop('Payment_Behaviour', axis=1, inplace=True)



In [10]:
for col in df.select_dtypes(include='object').columns:
    print(f"\nFrecuencia de valores en {col}:")
    print(df[col].value_counts())


In [11]:
df['Credit_Mix'].nunique()


In [12]:
df['Credit_Mix'].unique()


In [13]:
df['Credit_Score'].head(10)


In [14]:
df['Credit_Score'].unique()

In [15]:
import matplotlib.pyplot as plt
import seaborn as sns

num_cols = df.select_dtypes(include=['int64','float64']).columns

for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot de {col}')
    plt.show()


In [16]:
import pandas as pd

def contar_outliers(df):
    outliers = {}

    for col in df.select_dtypes(include=['int64','float64']).columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        count = ((df[col] < lower) | (df[col] > upper)).sum()
        outliers[col] = count

    return pd.Series(outliers)

outliers_por_feature = contar_outliers(df)
print(outliers_por_feature)


In [17]:
for col in ["Total_EMI_per_month","Amount_invested_monthly","Monthly_Balance",'Outstanding_Debt','Delay_from_due_date','Annual_Income','Monthly_Inhand_Salary','Changed_Credit_Limit','Credit_Utilization_Ratio']:
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR

    df[col] = df[col].clip(lower, upper)


In [18]:
import pandas as pd

def contar_outliers(df):
    outliers = {}

    for col in df.select_dtypes(include=['int64','float64']).columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        count = ((df[col] < lower) | (df[col] > upper)).sum()
        outliers[col] = count

    return pd.Series(outliers)

outliers_por_feature = contar_outliers(df)
print(outliers_por_feature)


In [19]:
total_outliers = outliers_por_feature.sum()
print("Total de outliers:", total_outliers)


In [20]:
X = df.drop('Credit_Score', axis=1)
y = df['Credit_Score']

In [21]:
X.head()


In [22]:
df['Credit_Score'].value_counts()


In [23]:
from sklearn.preprocessing import LabelEncoder
import joblib

# columnas categóricas
cat_cols = X.select_dtypes(include='object').columns

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

# guardar los encoders
joblib.dump(encoders, 'label_encoders.joblib')


In [24]:
X.info()

In [25]:
for col, le in encoders.items():
    print(f"{col}: {le.classes_}")


In [26]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=10)

X_Selected = selector.fit_transform(X, y)


In [27]:
selected_features = X.columns[selector.get_support()]
print(selected_features)


In [28]:
from tensorflow.keras.utils import to_categorical

y = pd.Categorical(df['Credit_Score']).codes
y = to_categorical(y, num_classes=3)


In [29]:
from sklearn.decomposition import PCA
import pandas as pd

pca = PCA(n_components=5)

X_pca = pca.fit_transform(X_Selected)


In [30]:
X_pca = pd.DataFrame(X_pca, columns=['PC1','PC2','PC3','PC4','PC5'])
X_pca.head()


In [31]:
sum(pca.explained_variance_ratio_)



In [32]:
import joblib

joblib.dump(pca, 'modelo_pca.joblib')


In [33]:
from sklearn.preprocessing import MinMaxScaler
import joblib

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X_pca)

# guardar el scaler
joblib.dump(scaler, 'minmax_scaler.joblib')


In [34]:
X_scaled[:10]


In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.15, random_state=42, stratify=y
)


In [36]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

model = Sequential()

# capa de entrada (X_train tiene 5 features)
model.add(Input(shape=(5,)))

# capas ocultas
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))


# capa de salida (3 clases)
model.add(Dense(3, activation='softmax'))

model.summary()


In [37]:
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


checkpoint = ModelCheckpoint(
    'mejor_modelo.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,

    batch_size=32,
    validation_split=0.15,
    verbose=1,
    callbacks=[checkpoint]
)


In [38]:
from tensorflow.keras.models import load_model

modelo = load_model('mejor_modelo.keras')

In [39]:
import matplotlib.pyplot as plt

# accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Accuracy del modelo')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'])
plt.show()

# loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Loss del modelo')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'])
plt.show()
